# LiteEdit-Qwen — 01: Quantization Exploration

**Goal:** Run fp16 baseline vs INT4, measure VRAM and latency, save comparison figure.

**Outputs:**
- `results/figures/quant_comparison.png` — side-by-side visual
- `results/metrics/quant_quick.csv` — numbers for Table 1

**GPU:** T4 16 GB (Kaggle free tier) or better.

In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────
!pip install -q transformers accelerate peft bitsandbytes qwen-vl-utils
!pip install -q lpips torchmetrics pillow pandas pyyaml

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Project setup ─────────────────────────────────────────
import os, sys

# Clone repo if not already present (replace URL with yours after pushing)
if not os.path.exists('liteedit-qwen'):
    !git clone https://github.com/YOUR_USERNAME/liteedit-qwen.git

%cd liteedit-qwen
sys.path.insert(0, '.')

from paths import PATHS
print('Project root:', PATHS.results.parent)
print('Results dir :', PATHS.results)

In [ ]:
# ── Cell 3: Build demo datasets ───────────────────────────────────
!python data/dataset_bg.py --build_demo --data_dir data --n_images 5
print('Demo dataset ready.')

In [ ]:
# ── Cell 4: Load BASELINE (fp16) ──────────────────────────────────
import torch
from models.loader import load_model

torch.cuda.reset_peak_memory_stats()
model_fp16, processor, config = load_model('configs/baseline.yaml')
vram_load_fp16 = torch.cuda.max_memory_allocated() / 1e9
print(f'\n✓ fp16 loaded  |  VRAM at load: {vram_load_fp16:.2f} GB')

In [ ]:
# ── Cell 5: Run one edit (fp16) ───────────────────────────────────
import time
from PIL import Image
from tasks.bg_replace import BgReplaceTask

test_img = Image.open('data/test/bg_replace/input/img_0000.png').convert('RGB')
prompt   = 'Replace the background with a snowy mountain landscape'

task_fp16 = BgReplaceTask(model_fp16, processor, config)

torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
out_fp16, desc_fp16 = task_fp16.run(image=test_img, prompt=prompt)
lat_fp16  = time.perf_counter() - t0
vram_fp16 = torch.cuda.max_memory_allocated() / 1e9

print(f'Latency : {lat_fp16:.2f}s')
print(f'VRAM    : {vram_fp16:.2f} GB')
print(f'Output  : {desc_fp16[:120]}...')

In [ ]:
# ── Cell 6: Free fp16, load INT4 ─────────────────────────────────
import gc
del model_fp16, task_fp16
torch.cuda.empty_cache(); gc.collect()
print(f'After cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB')

torch.cuda.reset_peak_memory_stats()
model_int4, proc_int4, cfg_int4 = load_model('configs/quant_int4.yaml')
vram_load_int4 = torch.cuda.max_memory_allocated() / 1e9
print(f'\n✓ INT4 loaded  |  VRAM at load: {vram_load_int4:.2f} GB')

In [ ]:
# ── Cell 7: Run same edit (INT4) ──────────────────────────────────
task_int4 = BgReplaceTask(model_int4, proc_int4, cfg_int4)

torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
out_int4, desc_int4 = task_int4.run(image=test_img, prompt=prompt)
lat_int4  = time.perf_counter() - t0
vram_int4 = torch.cuda.max_memory_allocated() / 1e9

print(f'Latency : {lat_int4:.2f}s')
print(f'VRAM    : {vram_int4:.2f} GB')

In [ ]:
# ── Cell 8: LPIPS between fp16 and INT4 outputs ───────────────────
import lpips, numpy as np

def to_tensor(img):
    arr = np.array(img.resize((256,256))).astype(np.float32) / 127.5 - 1.0
    return torch.from_numpy(arr).permute(2,0,1).unsqueeze(0)

lpips_fn = lpips.LPIPS(net='alex').cuda()
with torch.no_grad():
    score = lpips_fn(to_tensor(out_fp16).cuda(), to_tensor(out_int4).cuda())

print('\n══════ COMPARISON ══════')
print(f'{"":20} {"fp16":>10} {"INT4":>10}')
print(f'{"Latency (s)":20} {lat_fp16:>10.2f} {lat_int4:>10.2f}')
print(f'{"Peak VRAM (GB)":20} {vram_fp16:>10.2f} {vram_int4:>10.2f}')
print(f'{"LPIPS vs fp16":20} {"0.000":>10} {score.item():>10.4f}')
print('══════════════════════')

In [ ]:
# ── Cell 9: Save figure → results/figures/ ────────────────────────
import matplotlib.pyplot as plt
from paths import PATHS

PATHS.figures.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(test_img);  axes[0].set_title('Input');                    axes[0].axis('off')
axes[1].imshow(out_fp16);  axes[1].set_title(f'fp16\n{lat_fp16:.1f}s · {vram_fp16:.1f}GB'); axes[1].axis('off')
axes[2].imshow(out_int4);  axes[2].set_title(f'INT4\n{lat_int4:.1f}s · {vram_int4:.1f}GB\nLPIPS={score.item():.4f}'); axes[2].axis('off')
plt.suptitle('Quantization comparison — background replacement', y=1.02)
plt.tight_layout()

fig_path = PATHS.figures / 'quant_comparison.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {fig_path}')

In [ ]:
# ── Cell 10: Save metrics CSV → results/metrics/ ──────────────────
import pandas as pd

PATHS.metrics.mkdir(parents=True, exist_ok=True)

rows = [
    {'config': 'baseline',   'task': 'bg_replace', 'latency_s': round(lat_fp16, 3),  'peak_vram_gb': round(vram_fp16, 3), 'lpips_vs_fp16': 0.0},
    {'config': 'quant_int4', 'task': 'bg_replace', 'latency_s': round(lat_int4, 3),  'peak_vram_gb': round(vram_int4, 3), 'lpips_vs_fp16': round(score.item(), 4)},
]
df = pd.DataFrame(rows)

csv_path = PATHS.metrics / 'quant_quick.csv'
df.to_csv(csv_path, index=False)
print(f'Metrics saved → {csv_path}')
df

## What to record for your paper

| | fp16 | INT4 |
|---|---|---|
| Load VRAM (GB) | ___ | ___ |
| Inference VRAM (GB) | ___ | ___ |
| Latency / image (s) | ___ | ___ |
| LPIPS vs fp16 | 0.000 | ___ |

**Next:** `02_step_sweep.ipynb` → quality vs step count per task.